In [0]:
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
landing_path = f"/Volumes/adb_caresync/landing/landing_volumn/{source_system}/{table_name}/{landing_timestamp}/"

# Check if landing path (or any parent directory) exists
try:
    dbutils.fs.ls(landing_path)
    path_exists = True
except Exception:
    path_exists = False

# Exit gracefully (no failure) when no new files are received
if not path_exists:
    print(f"No new files are received at {landing_path}")
    dbutils.notebook.exit(f"No new files are received at {landing_path}")

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

landing_df.display()

In [0]:
from pyspark.sql.functions import lit, current_timestamp

updates_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", current_timestamp())

In [0]:
updates_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")

In [0]:
%sql
SELECT * FROM adb_caresync.bronze.hospitals

In [0]:
%sql
select * from adb_caresync.bronze.lab_results